# Congress + 13F Smart-Money Flow Mosaic

    Combines congressional trading (House/Senate), institutional 13F holdings and flows with price/volume reaction to create a transparent "smart money" overlay signal.

    **Category:** Multi-source regulatory / alt-data / flow analysis

    **Primary API calls used:**
    - FMP / SEC congressional trades (equity.congress or fmp.get_house_trades / get_senate_trades)
    - Institutional holdings / 13F (regulatory.ownership, fmp.get_institutional_holders, sec)
    - Equity pricing and volume (eod / fmp)
    - Optional short interest / microstructure for context

    Purely public data. No user book or PMS required.

## Run Output

![71_congress_13f_smart_money_mosaic](../plots/71_congress_13f_smart_money_mosaic_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
import os
import json
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantjourney.sdk import QuantJourneyAPI

qj = QuantJourneyAPI.from_env()

START = os.getenv("QJ_EXAMPLE_START", "2018-01-01")
END = os.getenv("QJ_EXAMPLE_END", "2026-06-06")

plt.style.use("default")
plt.rcParams.update({
    "figure.figsize": (13, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
})


def unwrap(payload: Any) -> Any:
    if isinstance(payload, dict) and "data" in payload: payload = payload["data"]
    if isinstance(payload, dict) and "value" in payload: return payload["value"]
    return payload


def safe_call(label: str, fn, **kwargs) -> Any:
    try:
        out = fn(**kwargs)
        print(f"{label}: ok")
        return out
    except Exception as exc:
        print(f"{label}: unavailable")
        return None


def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None: return []
    if isinstance(value, list): return value
    if isinstance(value, dict):
        for k in ("rows", "data", "items", "holdings", "trades"):
            if isinstance(value.get(k), list): return value[k]
        return [value]
    return []


def price_frame(symbol: str, start: str = START, end: str = END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    df = pd.DataFrame(rows)
    if df.empty: raise RuntimeError(f"No prices for {symbol}")
    df["date"] = pd.to_datetime(df.get("date"))
    for c in ["close", "adjusted_close", "volume"]:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
    df["price"] = df.get("adjusted_close").fillna(df.get("close"))
    return df.dropna(subset=["price"]).sort_values("date").set_index("date")


def get_congress_trades(symbol: str) -> pd.DataFrame:
    house = as_rows(safe_call("House " + symbol, qj.fmp.get_house_trades, symbol=symbol))
    senate = as_rows(safe_call("Senate " + symbol, qj.fmp.get_senate_trades, symbol=symbol))
    df = pd.concat([pd.DataFrame(house).assign(source="house"),
                    pd.DataFrame(senate).assign(source="senate")], ignore_index=True)
    if not df.empty and "transactionDate" in df:
        df["event_date"] = pd.to_datetime(df["transactionDate"], errors="coerce")
    return df


def get_13f_holdings(symbol: str) -> pd.DataFrame:
    # Try multiple surfaces that expose 13F-style data
    for call in [
        lambda: qj.fmp.get_institutional_holders(symbol=symbol),
        lambda: qj.sec.get_institutional_holdings(symbol=symbol),
        lambda: qj.fmp.get_institutional_portfolio_summary(symbol=symbol),
    ]:
        payload = safe_call("13F " + symbol, call)
        rows = as_rows(payload)
        if rows:
            return pd.DataFrame(rows)
    return pd.DataFrame()


In [ ]:
symbols = ["AAPL", "MSFT", "NVDA", "AMZN"]

all_events = []
for sym in symbols:
    trades = get_congress_trades(sym)
    if not trades.empty:
        trades["symbol"] = sym
        all_events.append(trades[["symbol", "event_date", "source"]].dropna(subset=["event_date"]))

events = pd.concat(all_events, ignore_index=True) if all_events else pd.DataFrame(columns=["symbol", "event_date", "source"])
print("Congress events sample:\n", events.head(8))

# Price reaction around events (simple 21d forward)
reactions = []
px_cache = {}
for sym in symbols:
    if sym not in px_cache:
        try:
            px_cache[sym] = price_frame(sym)["price"]
        except Exception:
            continue
    p = px_cache[sym]
    sym_events = events[events["symbol"] == sym]
    for _, row in sym_events.iterrows():
        ed = row["event_date"]
        if ed in p.index:
            idx = p.index.get_loc(ed)
            if idx + 21 < len(p):
                fwd = p.iloc[idx + 21] / p.iloc[idx] - 1
                reactions.append({"symbol": sym, "event_date": ed, "source": row["source"], "fwd_21d": fwd})

react = pd.DataFrame(reactions)
if not react.empty:
    print("\nMean forward 21d return by source:\n", react.groupby("source")["fwd_21d"].mean().round(4))
    react.groupby("source")["fwd_21d"].hist(alpha=0.6, bins=20)
    plt.title("Forward 21d returns around congressional trades")
    plt.show()

# Quick 13F concentration view for one symbol
h13f = get_13f_holdings("AAPL")
print("\n13F sample rows for AAPL:", len(h13f))
if not h13f.empty:
    display(h13f.head(3))

print("\nMulti-source flow mosaic complete (congress + 13F + price reaction).")

## Notes

Candidate using only public regulatory (congress + 13F) + pricing data.
No PMS or live holdings. Symbol mapping for COT/13F and exact method names (fmp vs sec vs regulatory domain) should be treated as tenant configuration.
Retain request_ids for any production use of these signals.